In [1]:
import torch, torchio as tio
import torch.nn.functional as F

from torch.utils.data import DataLoader
from datasets import DTIContrastiveDataset
from models import Encoder
from pathlib import Path
from sklearn.model_selection import train_test_split

/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/python3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
home_dir = "/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training"

input_file = f"{home_dir}/tensor_paths.txt"
with open(input_file, "r") as f:
    final_paths = [Path(line.strip()) for line in f if line.strip()]

print(f"Loaded {len(final_paths)} paths from {input_file}")

_, val_paths = train_test_split(
    final_paths, test_size=0.2, random_state=42
)

Loaded 22989 paths from /home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/tensor_paths.txt


In [6]:
val_dataset = DTIContrastiveDataset(path_list=val_paths[:50])
val_loader = DataLoader(
    val_dataset,
    batch_size=50,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [4]:
@torch.inference_mode()
def knn_self_accuracy(encoder, loader, k=5, device='cuda', max_batches=None):
    """
    Self-consistency Top-k accuracy without labels.
    Each sample already has two augmented views from the Dataset.

    encoder : your trained model (encoder).eval()
    loader  : DataLoader that yields (view1, view2)
    k       : top-k neighbour threshold
    max_batches : if set, only evaluate first N batches (speed up)
    """
    encoder.eval()
    reps = [] 

    for b, (x1, x2) in enumerate(loader, 1):
        if max_batches and b > max_batches:
            break
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        z1 = F.normalize(encoder(x1), dim=1)
        z2 = F.normalize(encoder(x2), dim=1)
        reps.append(torch.cat([z1, z2], 0))

    z = torch.cat(reps, 0)
    sim = z @ z.t()
    sim.fill_diagonal_(-1)

    _, idx = sim.topk(k, dim=1)
    pos_hits = (idx == torch.arange(z.size(0), device=z.device).view(-1,1).bitwise_xor(1)).any(dim=1)
    topk_acc = pos_hits.float().mean().item()
    return topk_acc

In [7]:
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(layers=(3,4,6,3))
encoder.to(device)
ckpt = torch.load("./checkpoints/dti_best.pth", map_location=device, weights_only=True)
encoder.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)

top5 = knn_self_accuracy(encoder, val_loader, k=5, device=device, max_batches=64)
print(f"Self k-NN top-5 accuracy: {top5:.3f}")

/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/python3.10/lib/python3.10/site-packages/torchio/data/image.py:248: UserWarning: Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyTorch DataLoader with a torchio.SubjectsLoader so that the collated batch becomes a dictionary, as expected. See https://github.com/TorchIO-project/torchio/issues/1179 for more context about this issue.
  warnings.warn(message, stacklevel=1)


Self k-NN top-5 accuracy: 0.000
